# 端到端 RAG 评测：从“失败模式”出发构建评估流水线

目标不是只给一个分数，而是把评测拆成多个维度，并最终组合成一个**端到端的 pass/fail 标准**（是否“够格交付给客户/用户”）。

本 notebook 的主线：

- **Step 1**：从人工标注/用户反馈中总结失败模式 → 选择评测维度
- **Step 2**：为某个维度（Completeness）构建“干净的 benchmark”（不要把别的坏因混进来）
- **Step 3**：实现 LLM-as-judge 的 Completeness 指标（naive → advanced）
- **Step 4**：用 DeepEval 的 `FaithfulnessMetric` 评测“回答是否被检索上下文支持”
- **Step 5**：把 completeness + faithfulness 聚合成端到端评测流程

## Utils / 环境准备

请确保：

- 已设置 `OPENAI_API_KEY`
- 已安装：`pandas`、`scikit-learn`、`python-dotenv`、`langchain-openai`

本 notebook 的 factuality/faithfulness 评测使用 `deepeval`：

- 已安装 `deepeval`

In [1]:
import os
import random
from enum import Enum
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from sklearn.metrics import classification_report

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

from deepeval import evaluate
from deepeval.evaluate.configs import AsyncConfig, DisplayConfig
from deepeval.metrics import FaithfulnessMetric
from deepeval.test_case import LLMTestCase

load_dotenv()

DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
JUDGE_MODEL_NAME = os.getenv("OPENAI_JUDGE_MODEL", MODEL_NAME)

llm = ChatOpenAI(model=MODEL_NAME, temperature=0)
judge_llm = ChatOpenAI(model=JUDGE_MODEL_NAME, temperature=0)

DEEPEVAL_JUDGE_MODEL = os.getenv("DEEPEVAL_JUDGE_MODEL", JUDGE_MODEL_NAME)
DEEPEVAL_FAITHFULNESS_THRESHOLD = float(os.getenv("DEEPEVAL_FAITHFULNESS_THRESHOLD", "0.7"))


def batch(items: list[dict], size: int = 32) -> list[list[dict]]:
    return [items[i : i + size] for i in range(0, len(items), size)]


def safe_content(x: Any) -> str:
    if x is None:
        return ""
    return str(x)

In [2]:
# os.environ["HTTP_PROXY"] = "http://127.0.0.1:7890"
# os.environ["HTTPS_PROXY"] = "http://127.0.0.1:7890"
# os.environ["ALL_PROXY"] = "http://127.0.0.1:7890"

# Step 1：选择评测维度（从真实失败模式出发）

不要一上来就拍脑袋定义指标，而是先看真实的人工反馈/标注理由，看看用户到底在抱怨什么，然后再决定要评什么。

In [3]:
DATASET_URL = "https://ndownloader.figshare.com/files/53919875"
DATASET_PATH = DATA_DIR / "figshare_rag_feedback.csv"


def download_file(url: str, path: Path) -> Path:
    headers = {"User-Agent": "Mozilla/5.0"}
    if path.exists() and path.stat().st_size > 0:
        return path

    r = requests.get(url, timeout=60, headers=headers, allow_redirects=True)
    r.raise_for_status()
    path.write_bytes(r.content)
    return path


download_file(DATASET_URL, DATASET_PATH)

df = pd.read_csv(DATASET_PATH)
df.head(2)

,input,information_retrieved,output,annotation,annotation_reasoning
0,When was the original release of Johnny Turbo’...,Johnny Turbo’s Arcade: Express Raider Date And...,Johnny Turbo’s Arcade: Express Raider was orig...,good,NaN
1,Who is the CEO of Franklin Templeton Investments?,"Gregory Johnson\nCEO\nUpdated On : Sep 28, 201...",The CEO of Franklin Templeton Investments is G...,good,NaN


In [4]:
# 快速看一下 bad 样本的标注理由（人工反馈）
only_bad = df[df["annotation"] == "bad"].copy()
only_bad["annotation_reasoning"].sample(5, random_state=1).values.tolist()

["While the response provides a comprehensive overview of the visitor's experience, it includes excessive details that are not directly related to the question about lacking amenities. For instance, mentioning the noise from children, the bridge construction site, and the parking situation, although part of the overall experience, distracts from the core issue of missing amenities. This verbosity can confuse the reader and dilute the focus on the specific amenities that were lacking, making it harder to quickly identify the key points. As a customer support manager, I would advise keeping responses concise and focused on the customer's direct inquiry to ensure clarity and efficiency in communication.",
 'While the output appears comprehensive and plausible at first glance, it introduces an extra detail about playing classical music during training sessions that is not supported by the provided context. This addition could mislead new trainers into thinking that classical music is a rec

In [5]:
analyze_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是数据分析师。你会拿到一批用户对问答助手输出的差评理由。"
            "请总结最多 5 个最常见的问题类型（短语即可）。",
        ),
        ("user", "{reasons}"),
    ]
)

analyze_chain = analyze_prompt | llm

reasons_text = "\n".join(only_bad["annotation_reasoning"].tolist())
summary_msg = analyze_chain.invoke({"reasons": reasons_text})
summary_msg.content

'1. 引入未支持或未提及的额外细节（hallucination）  \n2. 关键信息遗漏导致回答不完整  \n3. 回答内容冗长，包含大量无关或多余信息（冗 verbosity）  \n4. 回答与上下文内容矛盾或错误（contradiction）  \n5. 回答细节不准确或数字、结构等信息错误'

原版 notebook 给出的一个典型归类是：

1. **Incomplete**：缺少关键点
2. **Inconcise**：冗长、不够简洁
3. **Hallucinations**：胡编/事实错误
4. **Contradictions**：与上下文矛盾

后面我们用“分而治之”的方式：先把某一个维度（Completeness）做对、做稳，再逐步加入别的维度并聚合。

# Step 2：为 Completeness 构建“干净的 benchmark”

关键点：并不是所有 `bad` 都是“缺少信息”。

如果我们直接把所有 bad 都拿来当 completeness 的负样本，就会把“幻觉/矛盾/表达差”等信号混进去，导致 completeness 指标训练/评估都跑偏。

所以原版做法是：让 LLM 先根据 `annotation_reasoning` 判断这个 bad 是否“主要问题是 completeness”，再筛出真正的 completeness negatives。

In [6]:
system_prompt_is_complete = (
    "你在判断一个问答助手输出的问题类型。\n"
    "任务：根据人工反馈（annotation reasoning），判断该回答的主要问题是否是 completeness（缺少关键点/关键信息）。\n"
    "如果主要问题是幻觉、矛盾、表达差等，则判定为 False。\n"
    "请返回 has_completeness_problem（布尔）以及简短 reasoning。"
)

USER_PROMPT_REASONING_EVAL = """Question:
{question}

Wrong Answer:
{output}

Annotation Reasoning:
{reasoning}
"""


class CompletenessRelated(BaseModel):
    reasoning: str
    has_completeness_problem: bool


judge_completeness_related = (
    ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt_is_complete),
            ("user", USER_PROMPT_REASONING_EVAL),
        ]
    )
    | judge_llm.with_structured_output(CompletenessRelated, method="json_schema", strict=True)
)

inputs = [
    {
        "question": safe_content(row["input"]),
        "output": safe_content(row["output"]),
        "reasoning": safe_content(row["annotation_reasoning"]),
    }
    for _, row in only_bad.iterrows()
]

responses: list[CompletenessRelated] = []
for chunk in batch(inputs, size=32):
    responses.extend(judge_completeness_related.batch(chunk))

only_bad["completeness_related"] = [r.has_completeness_problem for r in responses]
only_bad["completeness_related"].value_counts()

completeness_related
False    25
True     15
Name: count, dtype: int64

In [7]:
# 构建一个不那么失衡的小 benchmark：
# - 正样本：good（我们把它当 completeness=1）
# - 负样本：bad 且 completeness_related=True（completeness=0）

only_good = df[df["annotation"] == "good"].copy()

neg = only_bad[only_bad["completeness_related"]].sample(10, random_state=1)
pos = only_good.sample(20, random_state=1)

completeness_eval_df = pd.concat([pos, neg], ignore_index=True)
completeness_eval_df["annotation"] = completeness_eval_df["annotation"].replace({"good": 1, "bad": 0})

completeness_eval_df.shape, completeness_eval_df["annotation"].value_counts().to_dict()

/tmp/ipykernel_2068090/285268898.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  completeness_eval_df["annotation"] = completeness_eval_df["annotation"].replace({"good": 1, "bad": 0})


((30, 6), {1: 20, 0: 10})

# Step 3：Completeness 指标（LLM-as-judge）

原版先做一个 naive 版本：只判断 complete / incomplete。

然后你会看到它在这个 benchmark 上不够稳，于是再升级为“带上下文、带解释、三档评分”的 advanced 版本。

In [8]:
system_prompt_completeness_eval_naive = (
    "根据问题判断回答是否缺少重要信息。\n"
    "只输出 COMPLETE 或 INCOMPLETE。"
)

USER_PROMPT_ANSWER_EVAL = """Question:
{question}

Context:
{context}

Answer:
{answer}
"""


class ScoreValueBinary(int, Enum):
    INCOMPLETE = 0
    COMPLETE = 1


class CompletenessScoreBinary(BaseModel):
    score: ScoreValueBinary


naive_chain = (
    ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt_completeness_eval_naive),
            ("user", USER_PROMPT_ANSWER_EVAL),
        ]
    )
    | judge_llm.with_structured_output(CompletenessScoreBinary, method="json_schema", strict=True)
)

inputs = [
    {
        "question": safe_content(row["input"]),
        "context": safe_content(row.get("information_retrieved")),
        "answer": safe_content(row.get("output")),
    }
    for _, row in completeness_eval_df.iterrows()
]

naive_scores: list[CompletenessScoreBinary] = []
for chunk in batch(inputs, size=32):
    naive_scores.extend(naive_chain.batch(chunk))

completeness_eval_df["naive_method"] = [int(x.score.value) for x in naive_scores]

print(completeness_eval_df["naive_method"].value_counts().to_dict())
classification_report(
    completeness_eval_df["annotation"],
    completeness_eval_df["naive_method"],
    output_dict=True,
    zero_division=0,
)

{1: 24, 0: 6}


{'0': {'precision': 1.0, 'recall': 0.6, 'f1-score': 0.75, 'support': 10.0},
 '1': {'precision': 0.8333333333333334,
  'recall': 1.0,
  'f1-score': 0.9090909090909091,
  'support': 20.0},
 'accuracy': 0.8666666666666667,
 'macro avg': {'precision': 0.9166666666666667,
  'recall': 0.8,
  'f1-score': 0.8295454545454546,
  'support': 30.0},
 'weighted avg': {'precision': 0.888888888888889,
  'recall': 0.8666666666666667,
  'f1-score': 0.856060606060606,
  'support': 30.0}}

naive 方法的问题在于：它没有明确“完整答案应该包含哪些要点”，也没有强制它基于上下文逐条核对。

所以原版升级为 advanced 方法：

- 先列出“完整答案应包含的关键点”
- 再检查回答是否覆盖
- 输出解释 + 0/1/2 三档评分

In [9]:
system_prompt_completeness_eval_advanced = (
    "你在评估回答是否完整（completeness）。你会看到 Question、Supporting Context、Answer。\n\n"
    "步骤：\n"
    "1) 先基于 Question + Context 列出一个完整答案应该包含的关键点（心里列，不用输出太长）。\n"
    "2) 检查 Answer 是否覆盖这些关键点。\n\n"
    "输出：先给出简短 reasoning，然后给 score：\n"
    "0 - Not complete：缺关键点/关键部分没回答。\n"
    "1 - Partially complete：都提到了但不充分/支持弱。\n"
    "2 - Fully complete：全面且由 Context 支持。"
)


class ScoreValueTri(int, Enum):
    INCOMPLETE = 0
    PARTIALLY_COMPLETE = 1
    COMPLETE = 2


class CompletenessScoreTri(BaseModel):
    reasoning: str
    score: ScoreValueTri


advanced_chain = (
    ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt_completeness_eval_advanced),
            ("user", USER_PROMPT_ANSWER_EVAL),
        ]
    )
    | judge_llm.with_structured_output(CompletenessScoreTri, method="json_schema", strict=True)
)

adv_scores: list[CompletenessScoreTri] = []
for chunk in batch(inputs, size=32):
    adv_scores.extend(advanced_chain.batch(chunk))

completeness_eval_df["advanced_method"] = [1 if int(x.score.value) > 1 else 0 for x in adv_scores]
completeness_eval_df["advanced_reasoning"] = [x.reasoning for x in adv_scores]

print(completeness_eval_df["advanced_method"].value_counts().to_dict())
classification_report(
    completeness_eval_df["annotation"],
    completeness_eval_df["advanced_method"],
    output_dict=True,
    zero_division=0,
)

{1: 19, 0: 11}


{'0': {'precision': 0.7272727272727273,
  'recall': 0.8,
  'f1-score': 0.7619047619047619,
  'support': 10.0},
 '1': {'precision': 0.8947368421052632,
  'recall': 0.85,
  'f1-score': 0.8717948717948718,
  'support': 20.0},
 'accuracy': 0.8333333333333334,
 'macro avg': {'precision': 0.8110047846889952,
  'recall': 0.825,
  'f1-score': 0.8168498168498168,
  'support': 30.0},
 'weighted avg': {'precision': 0.838915470494418,
  'recall': 0.8333333333333334,
  'f1-score': 0.8351648351648353,
  'support': 30.0}}

# Step 4：Faithfulness / Factuality（使用 Deepeval）

这一节按原仓库的学习目标：对“回答是否被检索上下文支持”做自动化评测。

我们用 `deepeval` 的 `FaithfulnessMetric` 来给每条样本打分：

- 输入：question + retrieved context + answer
- 输出：faithfulness score（0~1）与二值结果

In [10]:
system_prompt_is_factually_correct = (
    "你在判断一个问答助手输出的问题类型。\n"
    "任务：根据人工反馈（annotation reasoning），判断该回答的主要问题是否是 factual correctness（幻觉/事实错误/与上下文矛盾）。\n"
    "如果主要问题是 completeness、表达差等，则判定为 False。\n"
    "请返回 has_factual_correctness_problem（布尔）以及简短 reasoning。"
)


class FactualRelated(BaseModel):
    reasoning: str
    has_factual_correctness_problem: bool


judge_factual_related = (
    ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt_is_factually_correct),
            ("user", USER_PROMPT_REASONING_EVAL),
        ]
    )
    | judge_llm.with_structured_output(FactualRelated, method="json_schema", strict=True)
)

inputs = [
    {
        "question": safe_content(row["input"]),
        "output": safe_content(row["output"]),
        "reasoning": safe_content(row["annotation_reasoning"]),
    }
    for _, row in only_bad.iterrows()
]

fact_responses: list[FactualRelated] = []
for chunk in batch(inputs, size=32):
    fact_responses.extend(judge_factual_related.batch(chunk))

only_bad["factual_related"] = [r.has_factual_correctness_problem for r in fact_responses]
only_bad["factual_related"].value_counts()

factual_related
True     20
False    20
Name: count, dtype: int64

In [11]:
# 构建 factuality 的评测集：正样本来自 good，负样本来自 bad 且 factual_related=True

neg = only_bad[only_bad["factual_related"]]
pos = only_good.sample(len(neg), random_state=1)

factual_eval_df = pd.concat([pos, neg], ignore_index=True)
factual_eval_df["annotation"] = factual_eval_df["annotation"].replace({"good": 1, "bad": 0})

factual_eval_df.shape, factual_eval_df["annotation"].value_counts().to_dict()

/tmp/ipykernel_2068090/2794998961.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  factual_eval_df["annotation"] = factual_eval_df["annotation"].replace({"good": 1, "bad": 0})


((40, 7), {1: 20, 0: 20})

下面用 `deepeval` 的 `FaithfulnessMetric` 在 `factual_eval_df` 上跑一遍，并把评分转成二值标签，最后用 `classification_report` 看一下和人工标注的对齐情况。

In [12]:
DEEPEVAL_MAX_CONCURRENT = 100

faithfulness_metric = FaithfulnessMetric(
    threshold=DEEPEVAL_FAITHFULNESS_THRESHOLD,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=True,
)

# 构造 test cases（批量）
test_cases = [
    LLMTestCase(
        input=safe_content(row["input"]),
        actual_output=safe_content(row["output"]),
        retrieval_context=[safe_content(row.get("information_retrieved"))],
    )
    for _, row in factual_eval_df.iterrows()
]

# 用官方 evaluate(...) 批量评测，并由 async_config 控制并发
result = evaluate(
    test_cases=test_cases,
    metrics=[faithfulness_metric],
    async_config=AsyncConfig(run_async=True, max_concurrent=DEEPEVAL_MAX_CONCURRENT),
    display_config=DisplayConfig(
        show_indicator=True,
        print_results=False,
        inspect_after_run=False,
    ),
)

scores: list[float] = []
reasons: list[str] = []

for tr in result.test_results:
    md = (tr.metrics_data or [None])[0]
    scores.append(float(md.score) if md and md.score is not None else 0.0)
    reasons.append(safe_content(md.reason) if md else "")

factual_eval_df["deepeval_faithfulness"] = scores
factual_eval_df["deepeval_faithfulness_reason"] = reasons
factual_eval_df["deepeval_faithfulness_binary"] = [
    1 if s >= DEEPEVAL_FAITHFULNESS_THRESHOLD else 0 for s in scores
]

print({"rows": len(factual_eval_df), "max_concurrent": DEEPEVAL_MAX_CONCURRENT})
print(factual_eval_df["deepeval_faithfulness_binary"].value_counts().to_dict())
classification_report(
    factual_eval_df["annotation"],
    factual_eval_df["deepeval_faithfulness_binary"],
    output_dict=True,
    zero_division=0,
)

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1-mini, strict=False, async_mode=True)...

Output()

⚠ WARNING: No hyperparameters logged.
» ]8;id=2259488;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 18.56s | token cost: 0.08697919999999999 USD)
» Test Results (40 total tests):
   » Pass Rate: 72.5% | Passed: 29 | Failed: 11

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

{'rows': 40, 'max_concurrent': 100}
{1: 29, 0: 11}


{'0': {'precision': 0.45454545454545453,
  'recall': 0.25,
  'f1-score': 0.3225806451612903,
  'support': 20.0},
 '1': {'precision': 0.4827586206896552,
  'recall': 0.7,
  'f1-score': 0.5714285714285714,
  'support': 20.0},
 'accuracy': 0.475,
 'macro avg': {'precision': 0.46865203761755486,
  'recall': 0.475,
  'f1-score': 0.4470046082949308,
  'support': 40.0},
 'weighted avg': {'precision': 0.4686520376175548,
  'recall': 0.475,
  'f1-score': 0.44700460829493077,
  'support': 40.0}}

# Step 5：聚合成端到端评测流水线（pass/fail）

这一节按原仓库：只取两条主线指标来做最终 `good/bad`：

- completeness：用 advanced completeness judge（0/1/2）
- factuality/faithfulness：用 Deepeval 的 faithfulness score（0~1）

并设置阈值：只要任一维度不过线，就判定最终为 bad。

In [13]:
FACTUALITY_THRESHOLD = DEEPEVAL_FAITHFULNESS_THRESHOLD
COMPLETENESS_THRESHOLD = 2  # advanced score 0/1/2 中，要求达到 2
DEEPEVAL_MAX_CONCURRENT = 100

PIPELINE_DF = df.copy()
# PIPELINE_DF = df.sample(200, random_state=1).copy()

# 1) completeness（用 advanced judge 批量跑）

completeness_inputs = [
    {
        "question": safe_content(row["input"]),
        "context": safe_content(row.get("information_retrieved")),
        "answer": safe_content(row.get("output")),
    }
    for _, row in PIPELINE_DF.iterrows()
]

completeness_scores: list[int] = []
for chunk in batch(completeness_inputs, size=32):
    objs = advanced_chain.batch(chunk)
    completeness_scores.extend([int(o.score.value) for o in objs])

PIPELINE_DF["completeness_score"] = completeness_scores

# 2) factuality/faithfulness（用官方 evaluate(...) 批量评测）
faithfulness_metric_pipeline = FaithfulnessMetric(
    threshold=FACTUALITY_THRESHOLD,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=False,
)

test_cases = [
    LLMTestCase(
        input=safe_content(row["input"]),
        actual_output=safe_content(row.get("output")),
        retrieval_context=[safe_content(row.get("information_retrieved"))],
    )
    for _, row in PIPELINE_DF.iterrows()
]

res = evaluate(
    test_cases=test_cases,
    metrics=[faithfulness_metric_pipeline],
    async_config=AsyncConfig(run_async=True, max_concurrent=DEEPEVAL_MAX_CONCURRENT),
    display_config=DisplayConfig(
        show_indicator=True,
        print_results=False,
        inspect_after_run=False,
    ),
)

PIPELINE_DF["faithfulness_score"] = [
    float((tr.metrics_data or [None])[0].score)
    if (tr.metrics_data and tr.metrics_data[0].score is not None)
    else 0.0
    for tr in res.test_results
]

# 3) 聚合

def get_final_annotation(row: pd.Series) -> str:
    if float(row["faithfulness_score"]) < FACTUALITY_THRESHOLD:
        return "bad"
    if int(row["completeness_score"]) < COMPLETENESS_THRESHOLD:
        return "bad"
    return "good"


PIPELINE_DF["automated_annotation_pipeline"] = PIPELINE_DF.apply(get_final_annotation, axis=1)

print({"rows": len(PIPELINE_DF), "max_concurrent": DEEPEVAL_MAX_CONCURRENT})
classification_report(
    PIPELINE_DF["annotation"],
    PIPELINE_DF["automated_annotation_pipeline"],
    output_dict=True,
    zero_division=0,
)

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1-mini, strict=False, async_mode=True)...

Output()

⚠ WARNING: No hyperparameters logged.
» ]8;id=2259490;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 103.09s | token cost: 0.16042840000000003 USD)
» Test Results (80 total tests):
   » Pass Rate: 82.5% | Passed: 66 | Failed: 14

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

{'rows': 80, 'max_concurrent': 100}


{'bad': {'precision': 0.5909090909090909,
  'recall': 0.65,
  'f1-score': 0.6190476190476191,
  'support': 40.0},
 'good': {'precision': 0.6111111111111112,
  'recall': 0.55,
  'f1-score': 0.5789473684210527,
  'support': 40.0},
 'accuracy': 0.6,
 'macro avg': {'precision': 0.601010101010101,
  'recall': 0.6000000000000001,
  'f1-score': 0.5989974937343359,
  'support': 80.0},
 'weighted avg': {'precision': 0.601010101010101,
  'recall': 0.6,
  'f1-score': 0.5989974937343359,
  'support': 80.0}}

# Summary

现在得到的是一个“端到端评测流水线”的最小雏形：

- 先用真实反馈确定要评哪些维度（本 notebook：completeness + faithfulness）
- 对每个维度，先构建干净 benchmark，再做自动化指标（completeness 用 LLM-as-judge；faithfulness 用 DeepEval）
- 最后用阈值把多个维度聚合成 `good/bad`

在真实项目里，你通常会把 `df` 换成自己的线上日志（input / retrieved_context / output / human feedback），然后复用同样的流程来对比不同版本的 RAG 系统。